# 분산분석(ANOVA) 실습 정리

In [ ]:
# (필요 시 아래 설치 명령의 주석을 해제하고 먼저 실행하세요)
# !pip install pandas numpy scipy statsmodels matplotlib seaborn pingouin scikit-posthocs

# 데이터 분석
import numpy as np
import scipy as sp
import pandas as pd
from pathlib import Path

# 통계 분석
from scipy.stats import levene
from statsmodels.formula.api import ols            # 회귀식 적합(집단의 분해)
from statsmodels.stats.anova import anova_lm        # 분산분석표
import statsmodels.api as sm
import statsmodels.stats.oneway as anova_oneway     # Welch's ANOVA (statsmodels)
from statsmodels.graphics.factorplots import interaction_plot
import scikit_posthocs as sph                        # 사후검정(Scheffe 등)
import pingouin as pg                                 # ANOVA / Welch / Games-Howell 등

# 시각화
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 출력
plt.rcParams['font.family'] = 'Malgun Gothic'    # 한글폰트를 맑은고딕(Windows)
plt.rcParams['axes.unicode_minus'] = False       # 마이너스기호 깨짐 방지

# Apple(Mac) 계열의 한글출력은 아래를 활성화(위는 비활성화)
# plt.rcParams['font.family'] = 'AppleGothic'
# plt.rcParams['axes.unicode_minus'] = False

print('pingouin Version :', pg.__version__)

# pingouin 버전에 따라 컬럼명이 'p-unc'/'p_unc'처럼 하이픈(-)과 언더바(_)로 다르게
# 나오는 경우가 있어, 실제 존재하는 컬럼명을 찾아주는 헬퍼 함수를 만들어 둡니다.
def find_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'{candidates} 중 일치하는 컬럼이 없습니다. 실제 컬럼: {list(df.columns)}')

### CSV 경로 설정

노트북이 `Analysis/practice/` 폴더에 있고, CSV 파일이 `Analysis/source/` 폴더에 있으므로
`Path.cwd()`(현재 노트북 실행 위치)를 기준으로 상위 폴더의 `source` 폴더를 가리키도록 설정합니다.

노트북을 다른 위치에서 실행할 계획이라면, 아래 `SOURCE_DIR`을 절대경로로 직접 지정해도 됩니다.
(예: `SOURCE_DIR = Path(r'C:\Users\사용자명\Desktop\Analysis\source')`)

In [ ]:
BASE_DIR = Path.cwd()              # 현재 노트북(practice 폴더) 위치
SOURCE_DIR = BASE_DIR.parent / 'source'   # Analysis/source 폴더

# 경로가 올바른지 확인
print('노트북 위치      :', BASE_DIR)
print('CSV 파일 폴더     :', SOURCE_DIR)
print('폴더 존재 여부    :', SOURCE_DIR.exists())
if SOURCE_DIR.exists():
    print('source 폴더 내 CSV 목록:', [f.name for f in SOURCE_DIR.glob('*.csv')])

## 1. 일원분산분석 (One-way ANOVA)

### 1.1 데이터 불러오기 및 탐색

In [ ]:
one_anova = pd.read_csv(SOURCE_DIR / 'one_ANOVA.csv')
one_anova.head()

In [ ]:
# 집단(conv)별 표본 수 확인
one_anova.value_counts('conv')

In [ ]:
# 집단별 만족도(satis) 분리
cu       = one_anova[one_anova['conv'] == 1]['satis']
gs25     = one_anova[one_anova['conv'] == 2]['satis']
emart24  = one_anova[one_anova['conv'] == 3]['satis']
circle_k = one_anova[one_anova['conv'] == 4]['satis']
ampm     = one_anova[one_anova['conv'] == 5]['satis']

### 1.2 등분산성 검정 (Levene's Test)

In [ ]:
# center 옵션에 따른 Levene 검정 비교 (mean / median / trimmed)
for center in ['mean', 'median', 'trimmed']:
    stat, pvalue = levene(cu, gs25, emart24, circle_k, ampm, center=center)
    print(f"[center='{center}'] Levene's test statistic: {stat:.4f}, p-value: {pvalue:.4f}")

### 1.3 일원분산분석 실행

In [ ]:
# statsmodels: 회귀식 적합 후 분산분석표 산출
model = ols(formula='satis ~ C(conv)', data=one_anova).fit()
anova_lm(model)

In [ ]:
# pingouin: 한 줄로 분산분석표 산출 (효과크기 등 추가 정보 포함)
one_anova_pg = pg.anova(data=one_anova, dv='satis', between='conv', detailed=True)
one_anova_pg

### 1.4 사후검정 (Scheffe)

In [ ]:
scheffe = sph.posthoc_scheffe(one_anova, val_col='satis', group_col='conv')
scheffe_sorted = pd.DataFrame(scheffe).sort_index().sort_index(axis=1)
scheffe_sorted

## 2. 이원분산분석 (Two-way ANOVA)

### 2.1 데이터 불러오기 및 탐색

In [ ]:
two_anova = pd.read_csv(SOURCE_DIR / 'two_ANOVA.csv')
print('shape:', two_anova.shape)
two_anova.head()

In [ ]:
# 요인 수준별 표본 수 확인
print(two_anova.value_counts('style'))
print()
print(two_anova.value_counts('location'))

### 2.2 종속변수 및 요인 정의

In [ ]:
income   = two_anova['income']     # 종속변수
style    = two_anova['style']       # 요인 1
location = two_anova['location']    # 요인 2

### 2.3 등분산성 검정 (Levene's Test)

In [ ]:
levene_test_result = levene(
    income[(style == 1) & (location == 1)],
    income[(style == 1) & (location == 2)],
    income[(style == 1) & (location == 3)],
    income[(style == 2) & (location == 1)],
    income[(style == 2) & (location == 2)],
    income[(style == 2) & (location == 3)],
    income[(style == 3) & (location == 1)],
    income[(style == 3) & (location == 2)],
    income[(style == 3) & (location == 3)],
)

print("Levene's Test Statistic:", levene_test_result.statistic)
print("p-value:", levene_test_result.pvalue)

> 원본 코드는 `income[style==1][location==1]`처럼 불리언 인덱싱을 연쇄적으로 사용했는데,
> `(style == 1) & (location == 1)`로 조건을 결합하는 방식이 더 안전하고 명확합니다.

### 2.4 주효과 검정

In [ ]:
two_aov = ols(formula='income ~ C(style) + C(location)', data=two_anova).fit()
anova_lm(two_aov, typ=3)

### 2.5 상호작용 효과 검정

In [ ]:
pg.anova(data=two_anova, dv='income', between=['style', 'location'], ss_type=3)

참고: statsmodels로 동일한 분석을 수행하는 방법

In [ ]:
two_aov_c = ols(formula='income ~ C(style) * C(location)', data=two_anova).fit()
anova_lm(two_aov_c, typ=3)

### 2.6 그룹별 평균 시각화

In [ ]:
# Barplot: style x location 평균 + 표준편차 에러바
plt.figure(figsize=(6, 4))
sns.barplot(data=two_anova, x='style', y='income', hue='location',
            palette='pastel', capsize=0.1, errorbar='sd')
plt.title('Income by Style and Location')
plt.ylabel('Mean Income')
plt.xlabel('Style')
plt.legend(title='Location')
plt.tight_layout()
plt.show()

# Interaction plot: style x location 상호작용 패턴
plt.figure(figsize=(6, 4))
sns.pointplot(data=two_anova, x='style', y='income', hue='location',
              dodge=True, markers=['o', 's', 'D'], capsize=0.1,
              err_kws={'linewidth': 1.5}, palette='pastel')
plt.title('Interaction Plot: Style x Location')
plt.ylabel('Income')
plt.xlabel('Style')
plt.legend(title='Location')
plt.tight_layout()
plt.show()

### 2.7 사후검정 - 상호작용이 유의한 경우 (단순주효과 분석)

> pingouin 버전에 따라 p-value 컬럼명이 `p-unc`(구버전) 또는 `p_unc`(신버전, 언더바)처럼
> 다르게 나올 수 있어, 위 설정 셀에서 만든 `find_col()` 헬퍼로 실제 존재하는 컬럼명을
> 자동으로 찾아 사용합니다.

In [ ]:
# 상호작용이 유의할 때는 각 style 수준별로 location에 대한 단순주효과를 검토합니다.
for s in two_anova['style'].unique():
    subset = two_anova[two_anova['style'] == s]
    print(f'\n▶▶▶ 단순 주효과: style = {s} ◀◀◀')

    simple_effect_anova = pg.anova(data=subset, dv='income', between='location', ss_type=3)
    p_col = find_col(simple_effect_anova, 'p-unc', 'p_unc')
    print('\n--- 단순 주효과 ANOVA (Location)')
    print(simple_effect_anova[['Source', 'F', p_col]].round(4))

    tukey_results = pg.pairwise_tukey(data=subset, dv='income', between='location')
    pt_col = find_col(tukey_results, 'p-tukey', 'p_tukey', 'p-corr', 'p_corr')
    print(f'\n--- 단순 주효과 Tukey HSD: style = {s}')
    print(tukey_results[['A', 'B', 'diff', pt_col]].round(4))

# 필요하다면 반대로 location 수준별 style 차이도 동일한 방식으로 확인할 수 있습니다.

## 3. 등분산 가정 위배 시 대안

### 3.1 상호작용이 유의하지 않은 경우 (예시 데이터)

In [ ]:
two_anovax = pd.read_csv(SOURCE_DIR / 'two_ANOVAx.csv')

pg.anova(data=two_anovax, dv='income', between=['style', 'location'], ss_type=3)

In [ ]:
# 요인별 사후검정 (Scheffe)
scheffe_stylex = sph.posthoc_scheffe(two_anovax, val_col='income', group_col='style')
scheffe_locationx = sph.posthoc_scheffe(two_anovax, val_col='income', group_col='location')

print('--- style 사후검정 ---')
display(scheffe_stylex)
print('--- location 사후검정 ---')
display(scheffe_locationx)

### 3.2 Robust ANOVA (이분산 하의 강건 회귀, HC3)

In [ ]:
two_anova1 = pd.read_csv(SOURCE_DIR / 'two_ANOVA1.csv')

model = ols('income ~ C(style) * C(location)', data=two_anova1).fit()
robust_model = ols('income ~ C(style) * C(location)', data=two_anova1).fit(cov_type='HC3')

print('--- Robust ANOVA 결과 (HC3) ---')
print(robust_model.summary())

In [ ]:
# 참고: 일반적인 Type III ANOVA 결과
print('--- (참고) Type III ANOVA (일반적인 분산분석) ---')
print(sm.stats.anova_lm(model, typ=3))

### 3.3 Welch's ANOVA (등분산 가정을 요구하지 않는 분산분석)

In [ ]:
# pingouin
welch_anova_results = pg.welch_anova(data=two_anova1, dv='income', between='location')
welch_anova_results

In [ ]:
# statsmodels
welch_anova_sm = anova_oneway.anova_oneway(
    data=two_anova1['income'],
    groups=two_anova1['location'],
    use_var='unequal',
    welch_correction=True,
)
print("=== Statsmodels Welch's ANOVA 결과 ===")
print(welch_anova_sm)

## 4. 이분산성 하의 사후분석 (Games-Howell)

### 4.1 상호작용이 유의한 경우

In [ ]:
two_anova1 = pd.read_csv(SOURCE_DIR / 'two_ANOVA1.csv')

# style과 location을 결합한 그룹 변수 group_combo 생성
two_anova1['group_combo'] = two_anova1['style'].astype(str) + '_' + two_anova1['location'].astype(str)
two_anova1.head()

In [ ]:
# Games-Howell 사후분석 실행 (등분산 가정 없이 사용 가능)
posthoc_results = pg.pairwise_gameshowell(data=two_anova1, dv='income', between='group_combo')
posthoc_results

In [ ]:
# 유의한 조합만 필터링
alpha = 0.05
significant_pairs = posthoc_results[posthoc_results['pval'] < alpha]
significant_pairs

### 4.2 상호작용이 유의하지 않은 경우 (예시 데이터)

In [ ]:
two_anovax2 = pd.read_csv(SOURCE_DIR / 'two_ANOVAx2.csv')

pg.anova(data=two_anovax2, dv='Y', between=['A', 'B'], ss_type=3)

In [ ]:
# Welch's ANOVA 재실시 + Games-Howell 사후분석
print(pg.welch_anova(data=two_anovax2, dv='Y', between='A'))

posthoc_two_anovax2 = pg.pairwise_gameshowell(data=two_anovax2, dv='Y', between='A')
posthoc_two_anovax2

## 5. 시각화 모음

### 5.1 그룹별 분포 (Boxplot)

In [ ]:
two_anova1 = pd.read_csv(SOURCE_DIR / 'two_ANOVA1.csv')
two_anova1['group_combo'] = two_anova1['style'].astype(str) + '_' + two_anova1['location'].astype(str)

plt.figure(figsize=(8, 4))
ax = sns.boxplot(data=two_anova1, x='group_combo', y='income', hue='style', palette='Set2')
sns.stripplot(data=two_anova1, x='group_combo', y='income', color='black', size=3, jitter=0.2, ax=ax)

plt.title('Style과 Location에 따른 Income 분포', fontsize=15)
plt.xlabel('Group (Style_Location)')
plt.ylabel('Income')
plt.legend(title='Style')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

> 원본 코드는 `group_combo`를 만든 `two_anova1`이 아니라 `two_anova`를 그래프에 사용하는 오류가
> 있었습니다. 위 코드에서는 `two_anova1`로 통일했습니다.

### 5.2 상호작용 플롯 (Interaction Plot)

In [ ]:
two_anova1['style'] = two_anova1['style'].astype('category')
two_anova1['location'] = two_anova1['location'].astype('category')

fig, ax = plt.subplots(figsize=(7, 4))
interaction_plot(
    x=two_anova1['style'],
    trace=two_anova1['location'],
    response=two_anova1['income'],
    ax=ax,
    ms=10,
)
ax.set_title('Interaction Plot of Income by Style and Location')
ax.set_xlabel('Style')
ax.set_ylabel('Mean Income')
ax.legend(title='Location')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 5.3 잔차 진단 플롯 (등분산성 / 정규성 확인)

In [ ]:
model = ols('income ~ C(style) * C(location)', data=two_anova1).fit()
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 잔차 vs 적합값 (등분산성 확인)
sns.scatterplot(x=fitted, y=residuals, ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')

# Q-Q 플롯 (정규성 확인)
sm.qqplot(residuals, line='45', fit=True, ax=axes[1])
axes[1].set_title('Normal Q-Q Plot')

# 잔차 히스토그램
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_title('Residuals Distribution')
axes[2].set_xlabel('Residuals')

plt.tight_layout()
plt.show()

### 5.4 그룹별 평균 비교 (Bar + Error, Violin Plot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, errorbar_type, label in zip(axes, ['sd', 'se', 'ci'], ['SD', 'SE', '95% CI']):
    sns.barplot(x='location', y='income', hue='style', data=two_anova1,
                errorbar=errorbar_type, palette='Set2', capsize=0.1, ax=ax)
    ax.set_title(f'평균 및 {label} 비교')
    ax.set_xlabel('Location')
    ax.set_ylabel(f'Income (Mean ± {label})')
    ax.legend(title='Style')
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.violinplot(x='location', y='income', hue='style', data=two_anova1, palette='Set2')
plt.title('그룹별 분포 비교 (Violin Plot)', fontsize=14)
plt.xlabel('Location')
plt.ylabel('Income Distribution')
plt.legend(title='Style')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 5.5 사후검정 결과 시각화 (Games-Howell)

In [ ]:
posthoc_results = pg.pairwise_gameshowell(data=two_anova1, dv='income', between='group_combo')

# 그룹 간 평균 차이 히트맵
heat_data = posthoc_results.pivot(index='A', columns='B', values='diff')

plt.figure(figsize=(7, 5))
sns.heatmap(heat_data, annot=True, fmt='.1f', cmap='coolwarm', center=0)
plt.title('사후분석 그룹 간 평균 차이 (diff)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# p-value 히트맵
piv = posthoc_results.pivot(index='A', columns='B', values='pval')

plt.figure(figsize=(6, 5))
sns.heatmap(piv, annot=True, cmap='RdYlBu_r', fmt='.3f', cbar_kws={'label': 'p-value'})
plt.title('Post-hoc Test p-values (Games-Howell)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 평균 차이(diff) ± 표준오차(se) 플롯
plt.figure(figsize=(8, 4))
plt.errorbar(
    x=range(len(posthoc_results)),
    y=posthoc_results['diff'],
    yerr=posthoc_results['se'],
    fmt='o', capsize=5, color='steelblue',
)
plt.axhline(0, color='gray', linestyle='--')

labels = posthoc_results['A'].astype(str) + ' - ' + posthoc_results['B'].astype(str)
plt.xticks(range(len(posthoc_results)), labels, rotation=45, ha='right')

plt.ylabel('Mean Difference (± SE)')
plt.title('Post-hoc Comparison (Games-Howell)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. (보완) 평균 비교표에 사후검정 그룹 표시 — CLD(a, b, c) 표기

원본 노트북에는 "scikit-posthocs를 사용한 CLD(Compact Letter Display) 최종 코드"라는
제목만 있고 실제 코드는 비어 있었습니다. 아래는 사후검정 p-value 행렬을 이용해
같은 문자를 공유하는 집단끼리는 통계적으로 유의한 차이가 없음을 나타내는
간단한 CLD를 생성하는 예시 코드입니다.

> ⚠️ 아래 알고리즘은 교육용으로 작성한 단순 탐욕(greedy) 알고리즘이며,
> R의 `multcompView` 패키지처럼 엄밀하게 최소 문자 수를 보장하지는 않습니다.
> 정식 보고서에 사용할 때는 결과를 반드시 눈으로 다시 확인하세요.

In [ ]:
import string

def get_cld(pvalue_matrix: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """사후검정 p-value 행렬(symmetric)로부터 간단한 CLD(a, b, c ...) 문자를 생성합니다.

    Parameters
    ----------
    pvalue_matrix : 정사각형 DataFrame. index/columns가 그룹명이며,
                     원소는 두 그룹 간 사후검정 p-value.
    alpha : 유의수준

    Returns
    -------
    DataFrame(columns=['group', 'cld'])
    """
    groups = sorted(pvalue_matrix.index.astype(str), key=lambda x: str(x))
    remaining = groups.copy()
    letters = {g: '' for g in groups}
    letter_idx = 0

    def not_significant(g1, g2):
        return pvalue_matrix.loc[g1, g2] >= alpha

    while remaining:
        letter = string.ascii_lowercase[letter_idx]
        base = remaining[0]
        same_group = [base]
        for g in remaining[1:]:
            if all(not_significant(g, m) for m in same_group):
                same_group.append(g)
        for g in same_group:
            letters[g] += letter
        remaining = [g for g in remaining if g not in same_group]
        letter_idx += 1

    return pd.DataFrame({'group': groups, 'cld': [letters[g] for g in groups]})


# 예시: 일원분산분석(conv 별 satis) Scheffe 사후검정 결과를 CLD로 표시
scheffe = sph.posthoc_scheffe(one_anova, val_col='satis', group_col='conv')
scheffe.index = scheffe.columns = scheffe.columns.astype(str)

cld_table = get_cld(scheffe, alpha=0.05)

mean_table = one_anova.groupby('conv')['satis'].mean().reset_index()
mean_table['conv'] = mean_table['conv'].astype(str)

summary = mean_table.merge(cld_table, left_on='conv', right_on='group').drop(columns='group')
summary